<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/todo/Chapter11_nanogpt_todo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 11 — nanoGPT — assemble the full model**

> ✍️ TODO notebook — fill in the `# YOUR CODE HERE` blanks, then check against the answer.

## Goal

Bolt everything together into one `GPT` module:

```
ids → Embedding → [Block × N] → final LayerNorm → LM head → logits
```

The full forward also computes the loss when given targets, and we add a `generate()` method to
sample text. This is a complete, trainable GPT in ~40 lines.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
# All components from Ch 04-10, compactly (self-contained)
class Head(nn.Module):
    def __init__(s,C,hs,bs):
        super().__init__()
        s.key=nn.Linear(C,hs,bias=False); s.query=nn.Linear(C,hs,bias=False); s.value=nn.Linear(C,hs,bias=False)
        s.register_buffer("tril",torch.tril(torch.ones(bs,bs))); s.scale=hs**-0.5
    def forward(s,x):
        B,T,C=x.shape; k,q,v=s.key(x),s.query(x),s.value(x)
        a=(q@k.transpose(-2,-1)*s.scale).masked_fill(s.tril[:T,:T]==0,float("-inf"))
        return F.softmax(a,-1)@v
class MHA(nn.Module):
    def __init__(s,C,nh,bs): super().__init__(); s.heads=nn.ModuleList([Head(C,C//nh,bs) for _ in range(nh)]); s.proj=nn.Linear(C,C)
    def forward(s,x): return s.proj(torch.cat([h(x) for h in s.heads],-1))
class FF(nn.Module):
    def __init__(s,C): super().__init__(); s.net=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x): return s.net(x)
class Block(nn.Module):
    def __init__(s,C,nh,bs): super().__init__(); s.sa=MHA(C,nh,bs); s.ff=FF(C); s.ln1=nn.LayerNorm(C); s.ln2=nn.LayerNorm(C)
    def forward(s,x): x=x+s.sa(s.ln1(x)); x=x+s.ff(s.ln2(x)); return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, C=64, n_head=4, n_layer=4, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, C)
        self.pos = nn.Embedding(block_size, C)
        self.blocks = nn.Sequential(*[Block(C, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(C)
        self.head = nn.Linear(C, vocab_size)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        # YOUR CODE HERE: embed (token + position), run blocks, final LN, project to logits
        x = ...
        x = self.blocks(x)
        logits = ...
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, ids, max_new_tokens):
        for _ in range(max_new_tokens):
            ids_cond = ids[:, -self.block_size:]
            logits, _ = self(ids_cond)
            # YOUR CODE HERE: softmax the last step, sample one token, append it
            probs = ...
            next_id = ...
            ids = torch.cat([ids, next_id], dim=1)
        return ids

model = GPT(vocab_size=65).to(device)
print("params:", sum(p.numel() for p in model.parameters()), "trainable")
logits, loss = model(torch.randint(0, 65, (4, 32)).to(device),
                     torch.randint(0, 65, (4, 32)).to(device))
print("logits:", logits.shape, "| loss:", round(loss.item(), 3))

✅ A complete GPT! It's untrained (random) for now. Next: the training loop.